# MPB LaTeX OCR - Kaggle Model2 Training

Model2 is the page-level pipeline: YOLO26 detects formula regions, crops them in reading order, and the preserved model1 OCR checkpoint decodes each crop to LaTeX.

This notebook supports three practical Kaggle workflows:

1. Reuse an existing model1 OCR checkpoint and train only the YOLO detector.
2. Train model1 OCR from a crop-to-LaTeX dataset, then train the detector.
3. Run the full page-to-LaTeX pipeline on attached page images.

In [ ]:
import json
import os
import shlex
import shutil
import subprocess
from pathlib import Path

def run(command, cwd=None):
    print(f"$ {command}")
    subprocess.run(command, shell=True, check=True, cwd=cwd)

run("nvidia-smi || true")
print("Attached Kaggle datasets:")
run("find /kaggle/input -maxdepth 3 -type d | sort | head -120")

## Get The Project

Set `REPO_URL` to your repository URL. If you uploaded the repo as a Kaggle dataset, copy it into `/kaggle/working/mpb-latex-ocr` before the install cell.

In [ ]:
REPO_URL = "https://github.com/Ackrome/mpb-latex-ocr.git"
UPLOADED_REPO_DIR = None  # Optional: Path("/kaggle/input/YOUR_REPO_DATASET/mpb-latex-ocr")
PROJECT_DIR = Path("/kaggle/working/mpb-latex-ocr")

def discover_uploaded_repo() -> Path | None:
    if UPLOADED_REPO_DIR is not None:
        return Path(UPLOADED_REPO_DIR)
    for pyproject in Path("/kaggle/input").rglob("pyproject.toml"):
        candidate = pyproject.parent
        if (candidate / "src" / "mpb_latex_ocr").exists():
            return candidate
    return None

if not PROJECT_DIR.exists():
    uploaded_repo = discover_uploaded_repo()
    if REPO_URL:
        run(f"git clone {REPO_URL} {PROJECT_DIR}")
    elif uploaded_repo and uploaded_repo.exists():
        shutil.copytree(uploaded_repo, PROJECT_DIR)
    else:
        raise ValueError("Set REPO_URL, UPLOADED_REPO_DIR, or attach the repo as a Kaggle dataset.")

SRC_DIR = PROJECT_DIR / "src"
os.environ["PYTHONPATH"] = f"{SRC_DIR}:{os.environ.get('PYTHONPATH', '')}"
print("PROJECT_DIR:", PROJECT_DIR)
print("SRC_DIR:", SRC_DIR)

## Install

Kaggle normally provides CUDA PyTorch. This installs the repo in editable mode and adds detector dependencies for Ultralytics YOLO.

In [ ]:
run("python -m pip install --upgrade pip", cwd=PROJECT_DIR)
run('python -m pip install --force-reinstall --no-deps -e ".[detector,dev]"', cwd=PROJECT_DIR)
run("python -m pip install lightning mlflow omegaconf pillow matplotlib numpy tqdm pytest 'ultralytics>=8.4.0' pyyaml opencv-python-headless kaggle kagglehub roboflow", cwd=PROJECT_DIR)
run("python scripts/kaggle_preflight.py", cwd=PROJECT_DIR)

import sys
sys.path.insert(0, str(SRC_DIR))

import torch
import mpb_latex_ocr
import mpb_latex_ocr.detection
print("package:", mpb_latex_ocr.__file__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Configure Inputs

Attach the Kaggle datasets from the right sidebar, enable a GPU accelerator, then run the whole notebook. The notebook prepares the known OCR datasets into one manifest and converts the synthetic detection COCO annotations into YOLO format automatically.

Expected attached datasets:

- `willcsc/mathwriting`
- `gregoryeritsyan/im2latex-230k`
- `shiva22btcse0007/25k-math-equation`
- `anismekacher/synthetic-mathemtical-expression-detection`

You only need a manual override when using different datasets or existing model artifacts.


In [ ]:
# Main switches. Defaults are end-to-end: train model1, train detector, run page inference.
TRAIN_MODEL1_OCR = True
TRAIN_MODEL2_DETECTOR = True
RUN_MODEL2_PAGE_INFERENCE = True
AUTO_DISCOVER_INPUTS = True
AUTO_PREPARE_MODEL2_INPUTS = True

# Attach these public Kaggle datasets from the notebook sidebar and use Run All.
# Set AUTO_DOWNLOAD_DATASETS=True only if you want Kaggle to try downloading missing public datasets.
AUTO_DOWNLOAD_DATASETS = False
KAGGLE_DATASET_SLUGS = [
    "willcsc/mathwriting",
    "gregoryeritsyan/im2latex-230k",
    "shiva22btcse0007/25k-math-equation",
    "anismekacher/synthetic-mathemtical-expression-detection",
]

# Roboflow downloads require a ROBOFLOW_API_KEY Kaggle secret or environment variable.
# Keep this empty when using the attached synthetic Kaggle detection dataset above.
ROBOFLOW_DATASETS = []

DETECTOR_MODEL_WEIGHTS = "yolo26n.pt"
DETECTOR_CLASS_NAME_HINTS = [
    "formula",
    "equation",
    "equations",
    "math",
    "mathematical-expression",
    "mathematical expressions",
]

# Optional manual override for detector training. Leave None for auto-preparation/discovery.
DETECTOR_DATA_YAML = None  # Example: Path("/kaggle/input/formula-detector/data.yaml")

# Optional manual override for model1 OCR training. Leave None for auto-preparation/discovery.
MODEL1_OCR_DATASET_ROOT = None  # Example: Path("/kaggle/input/im2latex-230k/PRINTED_TEX_230k")
MODEL1_MANIFEST_PATH = None
MODEL1_MAX_SAMPLES = 50000
MODEL1_EPOCHS = 5
MODEL1_IMAGE_HEIGHT = 160
MODEL1_IMAGE_WIDTH = 640
MODEL1_MAX_LABEL_LENGTH = 512
MODEL1_BATCH_SIZE = 8

# Required when reusing model1 instead of training it here.
# Point these to an attached Kaggle dataset or a previous notebook artifact.
EXISTING_MODEL1_CHECKPOINT = None  # Example: Path("/kaggle/input/model1-run/best.ckpt")
EXISTING_MODEL1_TOKENIZER = None  # Example: Path("/kaggle/input/model1-run/tokenizer.json")

# Required only when TRAIN_MODEL2_DETECTOR=False and you still want inference.
EXISTING_DETECTOR_WEIGHTS = None  # Example: Path("/kaggle/input/formula-detector/best.pt")

# Optional manual override for final page-level inference.
# Leave None to use the detector dataset validation images.
PAGE_IMAGE_SOURCE = None  # Example: Path("/kaggle/input/page-images")

# Outputs saved as Kaggle artifacts
RUN_ROOT = Path("/kaggle/working/latex-ocr-runs")
MODEL1_OUTPUT_DIR = RUN_ROOT / "baseline"
DETECTOR_OUTPUT_DIR = RUN_ROOT / "detection" / "yolo"
MODEL2_OUTPUT_DIR = RUN_ROOT / "model2_page_predict"
PREPARED_INPUT_DIR = RUN_ROOT / "prepared_inputs"
MANIFEST_PATH = Path("/kaggle/working/latex-ocr-manifest.csv")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("RUN_ROOT:", RUN_ROOT)


## Download Public Datasets

Kaggle datasets are downloaded without extra credentials in most Kaggle notebooks. Roboflow Universe downloads require adding a `ROBOFLOW_API_KEY` secret to the notebook.

In [ ]:
DATASET_CACHE_ROOT = Path("/kaggle/working/datasets")
KAGGLE_DATASET_CACHE = DATASET_CACHE_ROOT / "kaggle"
ROBOFLOW_DATASET_CACHE = DATASET_CACHE_ROOT / "roboflow"

def safe_name(value: str) -> str:
    return value.replace("/", "__").replace(" ", "_")

def path_has_files(path: Path) -> bool:
    return path.exists() and any(child.is_file() for child in path.rglob("*"))

def run_optional(command: str, cwd: Path | None = None) -> bool:
    print(f"$ {command}")
    result = subprocess.run(command, shell=True, cwd=cwd)
    if result.returncode != 0:
        print(f"Command failed with exit code {result.returncode}; continuing so attached datasets can still be used.")
        return False
    return True

def get_kaggle_secret(name: str) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

def download_kaggle_dataset(slug: str) -> Path | None:
    try:
        import kagglehub

        hub_path = Path(kagglehub.dataset_download(slug))
        if path_has_files(hub_path):
            print(f"KaggleHub dataset available: {hub_path}")
            return hub_path
    except Exception as exc:
        print(f"KaggleHub download failed for {slug}: {exc}")

    target = KAGGLE_DATASET_CACHE / safe_name(slug)
    if path_has_files(target):
        print(f"Kaggle dataset already present: {target}")
        return target
    target.mkdir(parents=True, exist_ok=True)
    command = " ".join([
        "kaggle datasets download",
        "-d", shlex.quote(slug),
        "-p", shlex.quote(str(target)),
        "--unzip",
    ])
    return target if run_optional(command) and path_has_files(target) else None

def download_roboflow_dataset(spec: dict) -> Path | None:
    api_key = os.environ.get("ROBOFLOW_API_KEY") or get_kaggle_secret("ROBOFLOW_API_KEY")
    if not api_key:
        print("Skipping Roboflow download: add a ROBOFLOW_API_KEY Kaggle secret to enable it.")
        return None

    target_root = ROBOFLOW_DATASET_CACHE / safe_name(spec["project"])
    if any(target_root.rglob("data.yaml")):
        print(f"Roboflow dataset already present: {target_root}")
        return target_root

    try:
        from roboflow import Roboflow

        version = (
            Roboflow(api_key=api_key)
            .workspace(spec["workspace"])
            .project(spec["project"])
            .version(int(spec["version"]))
        )
        for dataset_format in spec.get("formats", ["yolo26", "yolov8"]):
            try:
                location = target_root / dataset_format
                version.download(dataset_format, location=str(location))
                if any(location.rglob("data.yaml")):
                    return location
            except Exception as exc:
                print(f"Roboflow format {dataset_format!r} failed: {exc}")
    except Exception as exc:
        print(f"Roboflow download failed: {exc}")
    return None

if AUTO_DOWNLOAD_DATASETS:
    DATASET_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    downloaded = []
    for slug in KAGGLE_DATASET_SLUGS:
        downloaded.append(download_kaggle_dataset(slug))
    for spec in ROBOFLOW_DATASETS:
        downloaded.append(download_roboflow_dataset(spec))
    print("Downloaded/discovered dataset roots:")
    for path in downloaded:
        if path is not None:
            print(" -", path)
else:
    print("AUTO_DOWNLOAD_DATASETS=False; using attached datasets only.")

## Auto-Prepare And Discover Attached Kaggle Inputs

This cell is what makes the notebook Run-All friendly. It searches attached Kaggle inputs and the optional download cache, creates a combined model1 OCR manifest from the known OCR datasets, converts the synthetic detection COCO annotations to YOLO when needed, and fills the downstream path variables.


In [ ]:
import yaml

INPUT_ROOTS = [Path("/kaggle/input"), DATASET_CACHE_ROOT]
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}

def has_images(path: Path) -> bool:
    return path.exists() and any(child.suffix.lower() in IMAGE_EXTENSIONS for child in path.rglob("*") if child.is_file())

def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        data = yaml.safe_load(handle) or {}
    return data if isinstance(data, dict) else {}

def yolo_root(data_yaml_path: Path, data: dict) -> Path:
    raw_root = data.get("path")
    if raw_root:
        root = Path(raw_root)
        if not root.is_absolute():
            root = data_yaml_path.parent / root
        return root
    return data_yaml_path.parent

def yolo_split_path(data_yaml_path: Path, split: str) -> Path | None:
    data = load_yaml(data_yaml_path)
    raw_value = data.get(split)
    if isinstance(raw_value, list):
        raw_value = raw_value[0] if raw_value else None
    if not raw_value:
        return None
    path = Path(str(raw_value))
    if path.is_absolute():
        return path
    return yolo_root(data_yaml_path, data) / path

def detector_name_score(data: dict) -> int:
    names = data.get("names", [])
    if isinstance(names, dict):
        values = names.values()
    else:
        values = names
    hints = {name.strip().lower().replace("_", "-").replace(" ", "-") for name in DETECTOR_CLASS_NAME_HINTS}
    return sum(str(name).strip().lower().replace("_", "-").replace(" ", "-") in hints for name in values)

def discover_detector_data_yaml() -> Path | None:
    candidates = []
    for root in INPUT_ROOTS:
        if not root.exists():
            continue
        for path in root.rglob("data.yaml"):
            data = load_yaml(path)
            if all(key in data for key in ("train", "val", "names")):
                train_dir = yolo_split_path(path, "train")
                val_dir = yolo_split_path(path, "val")
                score = 2 * int(train_dir is not None and has_images(train_dir)) + 2 * int(val_dir is not None and has_images(val_dir)) + detector_name_score(data)
                candidates.append((score, path))
    candidates.sort(key=lambda item: (-item[0], len(item[1].parts)))
    return candidates[0][1] if candidates else None

def discover_ocr_dataset_root() -> Path | None:
    paired_roots = []
    for input_root in INPUT_ROOTS:
        if not input_root.exists():
            continue
        for image_list in input_root.rglob("corresponding_png_images.txt"):
            root = image_list.parent
            if (root / "final_png_formulas.txt").exists() and has_images(root):
                paired_roots.append(root)
    if paired_roots:
        paired_roots.sort(key=lambda path: len(path.parts))
        return paired_roots[0]

    table_suffixes = {".csv", ".tsv", ".json", ".jsonl"}
    for input_root in INPUT_ROOTS:
        if not input_root.exists():
            continue
        for table in input_root.rglob("*"):
            if table.is_file() and table.suffix.lower() in table_suffixes and has_images(table.parent):
                return table.parent
    return None

def discover_page_image_source(data_yaml_path: Path | None) -> Path | None:
    if data_yaml_path is not None:
        for split in ("test", "val", "train"):
            split_dir = yolo_split_path(data_yaml_path, split)
            if split_dir is not None and has_images(split_dir):
                return split_dir
    for input_root in INPUT_ROOTS:
        if not input_root.exists():
            continue
        for path in input_root.iterdir():
            if path.is_dir() and has_images(path):
                return path
    return None

USER_SET_DETECTOR_DATA_YAML = DETECTOR_DATA_YAML is not None
USER_SET_MODEL1_MANIFEST_PATH = MODEL1_MANIFEST_PATH is not None
USER_SET_PAGE_IMAGE_SOURCE = PAGE_IMAGE_SOURCE is not None

if AUTO_DISCOVER_INPUTS:
    if DETECTOR_DATA_YAML is None:
        DETECTOR_DATA_YAML = discover_detector_data_yaml()
    if MODEL1_OCR_DATASET_ROOT is None:
        MODEL1_OCR_DATASET_ROOT = discover_ocr_dataset_root()
    if PAGE_IMAGE_SOURCE is None:
        PAGE_IMAGE_SOURCE = discover_page_image_source(DETECTOR_DATA_YAML)

PREPARED_INPUTS = {}
if AUTO_PREPARE_MODEL2_INPUTS:
    from mpb_latex_ocr.kaggle_model2_prepare import prepare_kaggle_model2_inputs

    PREPARED_INPUTS = prepare_kaggle_model2_inputs(
        input_roots=INPUT_ROOTS,
        output_dir=PREPARED_INPUT_DIR,
        manifest_path=MANIFEST_PATH,
        max_ocr_samples=MODEL1_MAX_SAMPLES if TRAIN_MODEL1_OCR else None,
    )
    print("Prepared Model2 inputs:")
    print(json.dumps(PREPARED_INPUTS, indent=2, sort_keys=True))

    if not USER_SET_MODEL1_MANIFEST_PATH and PREPARED_INPUTS.get("ocr_manifest"):
        MODEL1_MANIFEST_PATH = Path(PREPARED_INPUTS["ocr_manifest"])
    if not USER_SET_DETECTOR_DATA_YAML and PREPARED_INPUTS.get("detector_data_yaml"):
        DETECTOR_DATA_YAML = Path(PREPARED_INPUTS["detector_data_yaml"])
    if (
        not USER_SET_PAGE_IMAGE_SOURCE
        and not USER_SET_DETECTOR_DATA_YAML
        and PREPARED_INPUTS.get("page_image_source")
    ):
        PAGE_IMAGE_SOURCE = Path(PREPARED_INPUTS["page_image_source"])

DETECTOR_DATA_YAML = Path(DETECTOR_DATA_YAML) if DETECTOR_DATA_YAML is not None else None
MODEL1_OCR_DATASET_ROOT = Path(MODEL1_OCR_DATASET_ROOT) if MODEL1_OCR_DATASET_ROOT is not None else None
MODEL1_MANIFEST_PATH = Path(MODEL1_MANIFEST_PATH) if MODEL1_MANIFEST_PATH is not None else None
PAGE_IMAGE_SOURCE = Path(PAGE_IMAGE_SOURCE) if PAGE_IMAGE_SOURCE is not None else None

print("DETECTOR_DATA_YAML:", DETECTOR_DATA_YAML)
print("MODEL1_OCR_DATASET_ROOT:", MODEL1_OCR_DATASET_ROOT)
print("MODEL1_MANIFEST_PATH:", MODEL1_MANIFEST_PATH)
print("PAGE_IMAGE_SOURCE:", PAGE_IMAGE_SOURCE)


## Validate Detector Dataset

This is intentionally strict. If this cell fails, Kaggle is not pointing at a YOLO detector dataset yet.

In [ ]:
if TRAIN_MODEL2_DETECTOR:
    if DETECTOR_DATA_YAML is None or not DETECTOR_DATA_YAML.exists():
        raise FileNotFoundError("Attach a YOLO detector dataset or set DETECTOR_DATA_YAML manually.")
    data_yaml = yaml.safe_load(DETECTOR_DATA_YAML.read_text(encoding="utf-8"))
    print(data_yaml)
    for key in ("train", "val", "names"):
        if key not in data_yaml:
            raise ValueError(f"Detector data.yaml must include {key!r}")

    def normalize_detector_label(value: str) -> str:
        return value.strip().lower().replace("_", "-").replace(" ", "-")

    def detector_names_from_yaml(data: dict) -> list[str]:
        names = data.get("names", [])
        if isinstance(names, dict):
            return [str(names[key]) for key in sorted(names, key=lambda item: int(item))]
        return [str(name) for name in names]

    detector_class_names = detector_names_from_yaml(data_yaml)
    normalized_hints = {normalize_detector_label(name) for name in DETECTOR_CLASS_NAME_HINTS}
    DETECTOR_CLASS_NAME_FILTERS = [
        name for name in detector_class_names if normalize_detector_label(name) in normalized_hints
    ]
    print("Detector class names:", detector_class_names)
    print("Detector class filters for inference:", DETECTOR_CLASS_NAME_FILTERS or "all classes")
    print("Detector data.yaml OK:", DETECTOR_DATA_YAML)
else:
    DETECTOR_CLASS_NAME_FILTERS = []
    print("Skipping detector dataset validation because TRAIN_MODEL2_DETECTOR=False")

DETECTOR_CLASS_NAME_ARGS = " ".join(
    f"--class-name {shlex.quote(name)}" for name in DETECTOR_CLASS_NAME_FILTERS
)
PAGE_DETECTOR_CLASS_NAME_ARGS = " ".join(
    f"--detector-class-name {shlex.quote(name)}" for name in DETECTOR_CLASS_NAME_FILTERS
)
print("DETECTOR_CLASS_NAME_ARGS:", DETECTOR_CLASS_NAME_ARGS or "<none>")

## Train Or Reuse Model1 OCR

By default this trains model1 from the automatically prepared combined OCR manifest. Set `TRAIN_MODEL1_OCR = False` only when you attach an existing checkpoint and tokenizer.


In [ ]:
if TRAIN_MODEL1_OCR:
    if MODEL1_MANIFEST_PATH is not None and MODEL1_MANIFEST_PATH.exists():
        print("Using prepared OCR manifest:", MODEL1_MANIFEST_PATH)
    else:
        if MODEL1_OCR_DATASET_ROOT is None or not MODEL1_OCR_DATASET_ROOT.exists():
            raise FileNotFoundError("Attach the OCR Kaggle datasets or set MODEL1_OCR_DATASET_ROOT manually.")
        MODEL1_MANIFEST_PATH = MANIFEST_PATH
        prepare_cmd = " ".join([
            "python -m mpb_latex_ocr.prepare_manifest",
            f"--input-root {shlex.quote(str(MODEL1_OCR_DATASET_ROOT))}",
            f"--output {shlex.quote(str(MODEL1_MANIFEST_PATH))}",
            "--format auto",
            "--absolute-paths",
            f"--max-samples {MODEL1_MAX_SAMPLES}",
            "--val-fraction 0.05",
            "--test-fraction 0.05",
        ])
        run(prepare_cmd, cwd=PROJECT_DIR)

    run(f"head -5 {shlex.quote(str(MODEL1_MANIFEST_PATH))}")

    train_model1_cmd = " ".join([
        "python -m mpb_latex_ocr.train",
        "--config configs/train.yaml",
        "--config configs/model/deep_cnn.yaml",
        "--config configs/hardware/kaggle.yaml",
        "--config configs/datasets/kaggle_manifest.yaml",
        f"trainer.max_epochs={MODEL1_EPOCHS}",
        f"paths.manifest={shlex.quote(str(MODEL1_MANIFEST_PATH))}",
        f"data.image_height={MODEL1_IMAGE_HEIGHT}",
        f"data.image_width={MODEL1_IMAGE_WIDTH}",
        f"data.max_label_length={MODEL1_MAX_LABEL_LENGTH}",
        f"data.batch_size={MODEL1_BATCH_SIZE}",
        "data.augmentation_profile=handwriting",
        f"model.max_seq_len={MODEL1_MAX_LABEL_LENGTH}",
        f"generation.max_length={MODEL1_MAX_LABEL_LENGTH}",
        "tokenizer.force_rebuild=true",
    ])
    run(train_model1_cmd, cwd=PROJECT_DIR)
else:
    print("Skipping model1 OCR training; this notebook will use existing model1 artifacts.")


## Select Model1 Artifacts

Model2 needs a model1 checkpoint and tokenizer for page-level inference. If you trained model1 above, the notebook picks the newest local checkpoint. Otherwise set `EXISTING_MODEL1_CHECKPOINT` and `EXISTING_MODEL1_TOKENIZER`.

In [ ]:
def newest_checkpoint(directory: Path) -> Path | None:
    checkpoints = sorted(directory.glob("*.ckpt"), key=lambda path: path.stat().st_mtime)
    return checkpoints[-1] if checkpoints else None

if TRAIN_MODEL1_OCR:
    MODEL1_CHECKPOINT = newest_checkpoint(MODEL1_OUTPUT_DIR / "checkpoints")
    MODEL1_TOKENIZER = MODEL1_OUTPUT_DIR / "tokenizer.json"
else:
    MODEL1_CHECKPOINT = EXISTING_MODEL1_CHECKPOINT
    MODEL1_TOKENIZER = EXISTING_MODEL1_TOKENIZER

if RUN_MODEL2_PAGE_INFERENCE:
    if MODEL1_CHECKPOINT is None or not Path(MODEL1_CHECKPOINT).exists():
        raise FileNotFoundError("Set EXISTING_MODEL1_CHECKPOINT or enable TRAIN_MODEL1_OCR.")
    if MODEL1_TOKENIZER is None or not Path(MODEL1_TOKENIZER).exists():
        raise FileNotFoundError("Set EXISTING_MODEL1_TOKENIZER or enable TRAIN_MODEL1_OCR.")

print("MODEL1_CHECKPOINT:", MODEL1_CHECKPOINT)
print("MODEL1_TOKENIZER:", MODEL1_TOKENIZER)

## Train Model2 Detector

This trains the YOLO26 formula-region detector. It writes weights under `/kaggle/working/latex-ocr-runs/detection/yolo/formula_regions/weights/best.pt`.

In [ ]:
DETECTOR_EPOCHS = 50
DETECTOR_BATCH_SIZE = 8
DETECTOR_IMAGE_SIZE = 960
DETECTOR_RUN_NAME = "formula_regions"

if TRAIN_MODEL2_DETECTOR:
    train_detector_cmd = " ".join([
        "python -m mpb_latex_ocr.detect_train",
        "--config configs/detection/yolo_formula.yaml",
        f"data.yaml_path={DETECTOR_DATA_YAML}",
        f"model.weights={DETECTOR_MODEL_WEIGHTS}",
        f"model.image_size={DETECTOR_IMAGE_SIZE}",
        f"train.output_dir={DETECTOR_OUTPUT_DIR}",
        f"train.run_name={DETECTOR_RUN_NAME}",
        "train.device=0",
        f"train.epochs={DETECTOR_EPOCHS}",
        f"train.batch_size={DETECTOR_BATCH_SIZE}",
    ])
    run(train_detector_cmd, cwd=PROJECT_DIR)

if TRAIN_MODEL2_DETECTOR:
    DETECTOR_WEIGHTS = DETECTOR_OUTPUT_DIR / DETECTOR_RUN_NAME / "weights" / "best.pt"
else:
    DETECTOR_WEIGHTS = EXISTING_DETECTOR_WEIGHTS

if DETECTOR_WEIGHTS is None or not Path(DETECTOR_WEIGHTS).exists():
    raise FileNotFoundError("Set EXISTING_DETECTOR_WEIGHTS or enable TRAIN_MODEL2_DETECTOR.")
print("DETECTOR_WEIGHTS:", DETECTOR_WEIGHTS)

## Validate Detector On A Few Images

This runs YOLO detection only and saves formula crops plus `crops.jsonl`. Use it before full OCR to confirm the detector is finding the right regions.

In [ ]:
if RUN_MODEL2_PAGE_INFERENCE:
    if PAGE_IMAGE_SOURCE is None or not PAGE_IMAGE_SOURCE.exists():
        raise FileNotFoundError("Attach page images, set PAGE_IMAGE_SOURCE, or provide test/val images in detector data.yaml.")
    detect_only_dir = RUN_ROOT / "model2_detector_preview"
    detect_cmd = " ".join([
        "python -m mpb_latex_ocr.detect",
        f"--weights {DETECTOR_WEIGHTS}",
        f"--image {PAGE_IMAGE_SOURCE}",
        f"--output-dir {detect_only_dir / 'crops'}",
        f"--metadata-out {detect_only_dir / 'crops.jsonl'}",
        f"--image-size {DETECTOR_IMAGE_SIZE}",
        "--confidence 0.25",
        DETECTOR_CLASS_NAME_ARGS,
        "--device 0",
    ])
    run(detect_cmd, cwd=PROJECT_DIR)
    run(f"head -5 {detect_only_dir / 'crops.jsonl'} || true")
else:
    print("Skipping detector preview because RUN_MODEL2_PAGE_INFERENCE=False")

## Run Full Model2 Page Pipeline

This is the actual model2 inference path: page images -> YOLO boxes -> crops -> model1 OCR -> JSONL predictions.

In [ ]:
if RUN_MODEL2_PAGE_INFERENCE:
    predictions_out = MODEL2_OUTPUT_DIR / "predictions.jsonl"
    page_predict_cmd = " ".join([
        "python -m mpb_latex_ocr.page_predict",
        f"--detector-weights {DETECTOR_WEIGHTS}",
        f"--checkpoint {MODEL1_CHECKPOINT}",
        f"--tokenizer {MODEL1_TOKENIZER}",
        f"--image {PAGE_IMAGE_SOURCE}",
        f"--output-dir {MODEL2_OUTPUT_DIR}",
        f"--predictions-out {predictions_out}",
        f"--detector-image-size {DETECTOR_IMAGE_SIZE}",
        "--detector-confidence 0.25",
        PAGE_DETECTOR_CLASS_NAME_ARGS,
        "--detector-device 0",
        "--ocr-device cuda:0",
        f"--image-height {MODEL1_IMAGE_HEIGHT}",
        f"--image-width {MODEL1_IMAGE_WIDTH}",
        f"--max-generation-length {MODEL1_MAX_LABEL_LENGTH}",
    ])
    run(page_predict_cmd, cwd=PROJECT_DIR)
    print("predictions:", predictions_out)
    run(f"head -5 {predictions_out} || true")
else:
    print("Skipping full pipeline inference.")


## Inspect Model2 Outputs

This visual check overlays predicted boxes on page images and lists the decoded LaTeX rows. It is a sanity check, not a benchmark.

In [ ]:
import json
import random
import textwrap

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

VISUAL_PAGE_COUNT = 4
VISUAL_SEED = 42

predictions_path = MODEL2_OUTPUT_DIR / "predictions.jsonl"
if predictions_path.exists():
    rows = [json.loads(line) for line in predictions_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    by_page = {}
    for row in rows:
        by_page.setdefault(row["source_image_path"], []).append(row)

    pages = list(by_page)
    random.Random(VISUAL_SEED).shuffle(pages)
    for page_path in pages[:VISUAL_PAGE_COUNT]:
        page_rows = by_page[page_path]
        image = Image.open(page_path).convert("RGB")
        draw = ImageDraw.Draw(image)
        for idx, row in enumerate(page_rows):
            x1, y1, x2, y2 = [int(v) for v in row["bbox_xyxy"]]
            draw.rectangle((x1, y1, x2, y2), outline="red", width=3)
            draw.text((x1, max(0, y1 - 14)), str(idx), fill="red")

        plt.figure(figsize=(14, 10))
        plt.imshow(image)
        plt.axis("off")
        plt.title(page_path)
        plt.show()

        print("Detected formulas:")
        for idx, row in enumerate(page_rows):
            latex = "\n".join(textwrap.wrap(str(row.get("latex", "")), width=120))
            print(f"[{idx}] conf={row.get('confidence', 0):.3f} bbox={row.get('bbox_xyxy')}")
            print(latex)
            print("-" * 80)
else:
    print(f"No model2 predictions found yet: {predictions_path}")

## Artifact Checklist

Download these from Kaggle output artifacts:

- model1 checkpoint and tokenizer, if trained here: `/kaggle/working/latex-ocr-runs/baseline/`
- detector weights: `/kaggle/working/latex-ocr-runs/detection/yolo/formula_regions/weights/best.pt`
- model2 outputs: `/kaggle/working/latex-ocr-runs/model2_page_predict/`
- detector preview crops: `/kaggle/working/latex-ocr-runs/model2_detector_preview/`
